# M4.7 · Transformer ve GPT

Attention'ı LayerNorm, MLP ve residual bağlantılarla bir Transformer bloğuna, blokları da bir GPT'ye dönüştürüyorsun. GPT-2 small'ın 124M parametresini elle sayıyor, saf Python'da temperature/top-k örnekleme yazıyor, NumPy'da küçük bir GPT'nin ileri geçişini kuruyor, PyTorch'ta nanoGPT tarzı bir model yazıp ezberletiyor ve en sonda KV-cache ile üretimi yeniden hesaplamaya kıyasla hızlandırıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-7-transformer.html](https://m-gonul.github.io/pytorch-lab/m4-7-transformer.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_7_transformer.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def pc():\n        ensure(need(ns, \"block_params\")(768) == 7087872, \"block_params(768)\")\n        ensure(need(ns, \"gpt_params\")(50257, 1024, 768, 12) == 124439808, \"GPT-2 small = 124,439,808\")\n        ensure(need(ns, \"gpt_params\")(65, 256, 384, 6) == 65 * 384 + 256 * 384 + 6 * (12 * 384 ** 2 + 13 * 384) + 768, \"nanoGPT shakespeare-char\")\n    R.test(\"parametre sayıları\", pc)\n    R.test(\"probs: T = 1\", lambda: close(need(ns, \"probs\")([2, 1, 0]), [0.665241, 0.244728, 0.090031]))\n    R.test(\"probs: top-k = 2\", lambda: close(need(ns, \"probs\")([2, 1, 0], top_k=2), [0.731059, 0.268941, 0.0]))\n    R.test(\"probs: T = 0.5\", lambda: close(need(ns, \"probs\")([2, 1, 0], temperature=0.5), [0.866813, 0.117310, 0.015876], atol=1e-5))\n    R.test(\"probs: top-k sıraya bakmıyor\", lambda: close(need(ns, \"probs\")([0, 5, 1, 4], top_k=2), [0, 0.731059, 0, 0.268941]))\n\n    def smp():\n        s = need(ns, \"sample\")\n        ensure(s([2, 1, 0], 0.0) == 0 and s([2, 1, 0], 0.7) == 1 and s([2, 1, 0], 0.95) == 2, \"ters CDF: u=0→0, 0.7→1, 0.95→2\")\n        ensure(s([2, 1, 0], 0.95, top_k=2) == 1, \"top-k=2'de 3. token hiç seçilmez\")\n        ensure(s([0, 3, 1], 0.99, temperature=0) == 1, \"temperature 0 → argmax\")\n    R.test(\"sample\", smp)\n\n    def gen():\n        bigram = lambda toks: [3.0 if i == (toks[-1] + 1) % 5 else 0.0 for i in range(5)]\n        out = need(ns, \"generate\")(bigram, [0], 30, temperature=0, rng=random.Random(0))\n        ensure(out == [i % 5 for i in range(31)], \"greedy: 0,1,2,3,4,0,…\")\n        out = need(ns, \"generate\")(bigram, [0], 2000, temperature=1.0, rng=random.Random(1))\n        follow = sum(b == (a + 1) % 5 for a, b in zip(out, out[1:])) / 2000\n        want = math.exp(3) / (math.exp(3) + 4)\n        ensure(abs(follow - want) < 0.03, f\"takip oranı {follow:.3f}, beklenen ≈ {want:.3f}\")\n        return f\"T = 1'de 'sonraki' token oranı {follow:.3f} (teori {want:.3f})\"\n    R.test(\"generate\", gen)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef ref_forward(idx, P, H):\n    def ln(x, g, b):\n        return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + 1e-5) * g + b\n    def gelu(x):\n        return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))\n    B, T = idx.shape\n    out = []\n    for bi in range(B):\n        x = P[\"wte\"][idx[bi]] + P[\"wpe\"][:T]\n        d = x.shape[1]; dk = d // H\n        for p in P[\"blocks\"]:\n            h = ln(x, p[\"ln1_g\"], p[\"ln1_b\"])\n            qkv = h @ p[\"attn_w\"] + p[\"attn_b\"]\n            heads = []\n            for i in range(H):\n                q = qkv[:, i * dk:(i + 1) * dk]; k = qkv[:, d + i * dk:d + (i + 1) * dk]; v = qkv[:, 2 * d + i * dk:2 * d + (i + 1) * dk]\n                S = q @ k.T / np.sqrt(dk)\n                S[np.triu_indices(T, 1)] = -np.inf\n                A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)\n                heads.append(A @ v)\n            x = x + np.concatenate(heads, 1) @ p[\"proj_w\"] + p[\"proj_b\"]\n            x = x + gelu(ln(x, p[\"ln2_g\"], p[\"ln2_b\"]) @ p[\"fc_w\"] + p[\"fc_b\"]) @ p[\"fc2_w\"] + p[\"fc2_b\"]\n        out.append(ln(x, P[\"lnf_g\"], P[\"lnf_b\"]) @ P[\"wte\"].T)\n    return np.stack(out)\n\n\ndef params(rng, V, T, d, L):\n    def blk():\n        r = lambda *s: rng.normal(size=s) * 0.3\n        return dict(ln1_g=1 + r(d), ln1_b=r(d), attn_w=r(d, 3 * d), attn_b=r(3 * d), proj_w=r(d, d), proj_b=r(d),\n                    ln2_g=1 + r(d), ln2_b=r(d), fc_w=r(d, 4 * d), fc_b=r(4 * d), fc2_w=r(4 * d, d), fc2_b=r(d))\n    return dict(wte=rng.normal(size=(V, d)) * 0.3, wpe=rng.normal(size=(T, d)) * 0.3,\n                blocks=[blk() for _ in range(L)], lnf_g=1 + rng.normal(size=d) * 0.1, lnf_b=rng.normal(size=d) * 0.1)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"gpt_forward'da tek döngü (bloklar)\", lambda: max_loops(src, 1, func=\"gpt_forward\", why=\"head ve konumlar vektörel\"))\n    R.test(\"layer_norm, gelu döngüsüz\", lambda: (max_loops(src, 0, func=\"layer_norm\"), max_loops(src, 0, func=\"gelu\")))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(3)\n\n    def ln():\n        x = rng.normal(size=(2, 3, 6)); g = rng.normal(size=6); b = rng.normal(size=6)\n        y = need(ns, \"layer_norm\")(x, g, b)\n        close((y - b) / g, (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + 1e-5))\n    R.test(\"layer_norm\", ln)\n    R.test(\"gelu (tanh)\", lambda: close(need(ns, \"gelu\")(np.array([-1.0, 0, 1, 3])), [-0.158808, 0, 0.841192, 2.996363], atol=1e-5))\n    P = params(rng, 13, 10, 12, 3)\n    idx = rng.integers(0, 13, (2, 7))\n    R.test(\"3 blok, 3 head: başvuru ile aynı\", lambda: close(need(ns, \"gpt_forward\")(idx, P, 3), ref_forward(idx, P, 3)))\n\n    def causal():\n        idx2 = idx.copy(); idx2[:, 5:] = (idx2[:, 5:] + 1) % 13\n        a = need(ns, \"gpt_forward\")(idx, P, 3); b = need(ns, \"gpt_forward\")(idx2, P, 3)\n        close(a[:, :5], b[:, :5], what=\"ilk 5 konum\")\n    R.test(\"nedensellik\", causal)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def mk(seed=0, **kw):\n        torch.manual_seed(seed)\n        cfg = dict(vocab=29, block_size=40, d=32, n_layer=2, n_head=4); cfg.update(kw)\n        return need(ns, \"GPT\", \"sınıf\")(**cfg), cfg\n\n    def count():\n        m, c = mk()\n        V, T, d, L = c[\"vocab\"], c[\"block_size\"], c[\"d\"], c[\"n_layer\"]\n        got = sum(p.numel() for p in m.parameters())\n        want = V * d + T * d + L * (12 * d * d + 13 * d) + 2 * d\n        ensure(got == want, f\"{got} parametre; beklenen {want} (12d²+13d blok başına, tied head)\")\n        return f\"{got:,} parametre\"\n    R.test(\"parametre sayısı formülle aynı\", count)\n\n    def tie():\n        m, _ = mk()\n        ensure(isinstance(getattr(m, \"head\", None), nn.Linear) and m.head.weight is m.wte.weight, \"head.weight, wte.weight ile aynı nesne olmalı\")\n    R.test(\"weight tying\", tie)\n\n    def shapes():\n        m, _ = mk()\n        idx = torch.randint(0, 29, (3, 11))\n        logits, loss = m(idx)\n        ensure(tuple(logits.shape) == (3, 11, 29) and loss is None, \"targets yokken (logits (B,T,V), None)\")\n        _, loss = m(idx, idx)\n        ensure(loss.dim() == 0, \"loss skaler olmalı\")\n    R.test(\"çıktı şekilleri\", shapes)\n\n    def causal():\n        m, _ = mk(1)\n        m.eval()\n        idx = torch.randint(0, 29, (2, 20))\n        idx2 = idx.clone(); idx2[:, 12:] = (idx2[:, 12:] + 3) % 29\n        with torch.no_grad():\n            a, _ = m(idx); b, _ = m(idx2)\n        close(a[:, :12], b[:, :12], atol=1e-5, what=\"ilk 12 konum\")\n        ensure(not torch.allclose(a[:, 12:], b[:, 12:]), \"sonraki konumlar değişmeliydi\")\n    R.test(\"nedensellik\", causal)\n\n    def overfit():\n        text = \"pytorch ile gpt yaziyoruz; once elle, sonra vektorel.\"\n        chars = sorted(set(text))\n        ids = torch.tensor([chars.index(c) for c in text])[None]\n        m, _ = mk(2, vocab=len(chars), block_size=len(text))\n        opt = torch.optim.AdamW(m.parameters(), lr=3e-3)\n        for _ in range(250):\n            _, loss = m(ids[:, :-1], ids[:, 1:])\n            opt.zero_grad(); loss.backward(); opt.step()\n        ensure(loss.item() < 0.1, f\"250 adım sonra kayıp {loss.item():.3f}; < 0.1 bekleniyordu\")\n        m.eval()\n        out = ids[:, :1]\n        with torch.no_grad():\n            for _ in range(len(text) - 1):\n                out = torch.cat([out, m(out)[0][:, -1].argmax(-1, keepdim=True)], 1)\n        s = \"\".join(chars[i] for i in out[0].tolist())\n        return f\"kayıp {loss.item():.4f}; greedy: “{s}”\"\n    R.test(\"kısa metni ezberliyor\", overfit)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\n@torch.no_grad()\ndef greedy_naive(model, idx, n):\n    for _ in range(n):\n        idx = torch.cat([idx, model(idx)[:, -1].argmax(-1, keepdim=True)], 1)\n    return idx\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"forward_step: tek döngü\", lambda: max_loops(src, 1, func=\"forward_step\", why=\"yalnız bloklar\"))\n    R.test(\"generate_kv: tek döngü\", lambda: max_loops(src, 1, func=\"generate_kv\", why=\"yalnız adımlar\"))\n    TinyGPT = need(ns, \"TinyGPT\", \"sınıf\")\n    torch.manual_seed(0)\n    m = TinyGPT().eval()\n    for p in m.parameters():\n        p.data = p.data.double()\n    idx = torch.randint(0, 64, (2, 15))\n\n    def prefill():\n        with torch.no_grad():\n            logits, cache = need(ns, \"forward_step\")(m, idx, [None] * 4, 0)\n        close(logits, m(idx), what=\"prefill logit'leri\")\n        ensure(len(cache) == 4 and tuple(cache[0][0].shape) == (2, 4, 15, 16), \"cache: blok başına (K, V), (B, h, S, dk)\")\n    R.test(\"prefill = tam ileri geçiş\", prefill)\n\n    def steps():\n        full = m(idx)\n        with torch.no_grad():\n            lg, cache = need(ns, \"forward_step\")(m, idx[:, :6], [None] * 4, 0)\n            close(lg, full[:, :6], what=\"ilk 6\")\n            lg, cache = ns[\"forward_step\"](m, idx[:, 6:9], cache, 6)\n            close(lg, full[:, 6:9], what=\"3'lü parça (6–8)\")\n            for t in range(9, 15):\n                lg, cache = ns[\"forward_step\"](m, idx[:, t:t + 1], cache, t)\n                close(lg[:, 0], full[:, t], what=f\"konum {t}\")\n    R.test(\"artımlı adımlar her konumda tam geçişle aynı\", steps)\n\n    def gen():\n        out = need(ns, \"generate_kv\")(m, idx[:1, :5], 30)\n        ensure(torch.equal(out, greedy_naive(m, idx[:1, :5], 30)), \"cache'siz greedy ile farklı dizi\")\n    R.test(\"generate_kv = cache'siz greedy\", gen)\n    if \"generate_kv\" not in ns:\n        return\n    torch.manual_seed(0)\n    mf = TinyGPT(d=128, L=4, h=4).eval()\n    p = torch.randint(0, 64, (1, 16))\n    n = 200\n    t_naive = bench(lambda: greedy_naive(mf, p, n), repeat=1, number=1)\n    t_kv = bench(lambda: ns[\"generate_kv\"](mf, p, n), repeat=1, number=1)\n    R.table(f\"{n} token greedy, d = 128, 4 blok\", [\"yöntem\", \"süre\", \"token/s\"],\n            [[\"her adımda tüm diziyi yeniden hesapla\", fmt_time(t_naive), f\"{n / t_naive:,.0f}\"],\n             [\"senin KV-cache\", fmt_time(t_kv), f\"{n / t_kv:,.0f}\"]],\n            note=\"Naif yol adım t'de O(t) token işler → toplam O(n²). Cache bellek ister: 2 · blok · h · S · dk sayı.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_7_transformer as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

GPT, token + konum embedding'inin üstüne $L$ tane aynı yapıda blok koyar ve en sonda sözlük üzerinde logit üretir:

$$
\begin{aligned}
x_0 &= W_{te}[\text{idx}] + W_{pe}[0..T),\\
x &\leftarrow x + \operatorname{MHA}_{\text{causal}}(\operatorname{LN}_1(x)),\\
x &\leftarrow x + \operatorname{MLP}(\operatorname{LN}_2(x)),\quad \operatorname{MLP}(u)=W_2\,\operatorname{GELU}(W_1u),\\
\text{logits} &= \operatorname{LN}_f(x)\,W_{te}^\top.
\end{aligned}
$$

Son satırda çıkış katmanı ayrı bir matris değil, $W_{te}$'nin kendisi (weight tying): GPT-2 small'da 38.6M parametre tasarrufu.

Blok başına parametre: qkv $3d^2+3d$, proj $d^2+d$, MLP $8d^2+5d$, iki LN $4d$; toplam $12d^2+13d$.

**Örnekleme:** logit $z$, temperature $\tau$, top-k: en büyük $k$ logit dışındakileri $-\infty$ yap, sonra $p=\operatorname{softmax}(z/\tau)$.

**kv-cache:** üretimde her adımda yalnız son token'ın query'si gerekir; geçmiş token'ların key/value'larını saklarsan her adım tek bir token'lık iş olur.

## 1 · Elle

GPT-2 small'ın hesabı: embedding'ler $50257\cdot768+1024\cdot768=39{,}383{,}808$; bloklar $12\times7{,}087{,}872=85{,}054{,}464$; son LN $1{,}536$. Toplam $124{,}439{,}808$ — HuggingFace'in raporladığı sayı. Parametrelerin yaklaşık üçte biri embedding.

Top-k ile temperature birlikte çalışır: önce kes, sonra ölçekle ve normalize et.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Blok: LN₁ (γ, β), qkv Linear d→3d, proj d→d, LN₂, MLP d→4d→d; tüm Linear'larda bias var**

- `blk4`: $d=4$ için blok parametresi
- `blk768`: $d=768$ için blok parametresi
- `gpt2`: GPT-2 small: $V=50257$, $T=1024$, $d=768$, 12 blok, son LN, çıkış = wte (tied)

**Logit'ler $[2, 1, 0]$**

- `p1`: $T=1$: $p(\text{token}_1)$
- `topk`: top-k = 2: $p(\text{token}_1)$
- `temp`: temperature 0.5: $p(\text{token}_1)$

In [ ]:
cevaplar = dict(
    blk4=None,
    blk768=None,
    gpt2=None,
    p1=None,
    topk=None,
    temp=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"blk4":[244,0],"blk768":[7087872,0],"gpt2":[124439808,0],"p1":[0.665241,0.0005],"topk":[0.731059,0.0005],"temp":[0.866813,0.0005]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python:

- `block_params(d)` ve `gpt_params(V, T, d, n_layer)` — yukarıdaki yerleşim (tied çıkış, son LN dahil).
- `probs(logits, temperature=1.0, top_k=None)` → olasılık listesi. top-k'da tutulan token'lar en büyük $k$ logit (eşitlik yok kabul et); diğerleri 0.
- `sample(logits, u, temperature=1.0, top_k=None)` — `u ∈ [0, 1)` düzgün sayı; **ters CDF**: indeks sırasıyla birikimli olasılığı topla, `u < cdf` olan ilk indeksi döndür. `temperature == 0` ise argmax.
- `generate(next_logits, prompt, n, temperature, top_k, rng)` — `next_logits(tokens) → logits`; her adımda `rng.random()` ile örnekle, listeye ekle, `prompt + n` token döndür.

In [ ]:
%%writefile stage_b.py
import math
import random


def block_params(d):
    ...


def gpt_params(V, T, d, n_layer):
    ...


def probs(logits, temperature=1.0, top_k=None):
    ...


def sample(logits, u, temperature=1.0, top_k=None):
    ...


def generate(next_logits, prompt, n, temperature=1.0, top_k=None, rng=None):
    ...


print(gpt_params(50257, 1024, 768, 12))        # 124439808
print(probs([2, 1, 0], top_k=2))               # [0.731, 0.269, 0]
bigram = lambda toks: [3.0 if i == (toks[-1] + 1) % 5 else 0.0 for i in range(5)]
print(generate(bigram, [0], 8, temperature=0.7, rng=random.Random(0)))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy ile küçük bir GPT'nin ileri geçişi. Parametreler sözlükte (`P`), bloklar `P["blocks"]` listesinde; her blok: `ln1_g, ln1_b, attn_w (d,3d), attn_b, proj_w (d,d), proj_b, ln2_g, ln2_b, fc_w (d,4d), fc_b, fc2_w (4d,d), fc2_b`. Çarpım `x @ W`.

- `layer_norm(x, g, b, eps=1e-5)` — son eksende.
- `gelu(x)` — tanh yaklaşımı: $0.5x\big(1+\tanh(\sqrt{2/\pi}(x+0.044715x^3))\big)$.
- `gpt_forward(idx, P, n_head)` — `idx: (B, T)` → `(B, T, V)` logit. **Yalnız bloklar üzerinde tek döngü**; head'ler reshape ile.

In [ ]:
%%writefile stage_c.py
import numpy as np


def layer_norm(x, g, b, eps=1e-5):
    ...


def gelu(x):
    ...


def gpt_forward(idx, P, n_head):
    ...


rng = np.random.default_rng(0)
V, T, d, L = 11, 8, 8, 2
def blk():
    r = lambda *s: rng.normal(size=s) * 0.2
    return dict(ln1_g=np.ones(d), ln1_b=np.zeros(d), attn_w=r(d, 3 * d), attn_b=r(3 * d), proj_w=r(d, d), proj_b=r(d),
                ln2_g=np.ones(d), ln2_b=np.zeros(d), fc_w=r(d, 4 * d), fc_b=r(4 * d), fc2_w=r(4 * d, d), fc2_b=r(d))
P = dict(wte=rng.normal(size=(V, d)) * 0.2, wpe=rng.normal(size=(T, d)) * 0.2, blocks=[blk(), blk()], lnf_g=np.ones(d), lnf_b=np.zeros(d))
print(gpt_forward(np.array([[1, 2, 3, 4]]), P, n_head=2).shape)   # (1, 4, 11)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

nanoGPT tarzı:

- `GPT(vocab, block_size, d, n_layer, n_head)` — öznitelikler: `wte`, `wpe` (`nn.Embedding`), `blocks` (`nn.ModuleList`), `ln_f`, `head` (`nn.Linear(d, vocab, bias=False)`, ağırlığı `wte.weight` ile **aynı nesne**).
- Her blok: `ln1`, causal multi-head attention (`qkv` d→3d, `proj` d→d; `F.scaled_dot_product_attention(…, is_causal=True)` serbest), `ln2`, MLP (d→4d, `GELU(approximate="tanh")`, 4d→d). Tüm Linear'larda bias (head hariç).
- `forward(idx, targets=None)` → `(logits, loss)`; `targets` yoksa `loss = None`.

Kontrol parametre sayısını formülle, weight tying'i, nedenselliği (gelecek token değişince geçmiş logit'ler sabit) ve modelin kısa bir metni ezberleyebildiğini sınar.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


class Block(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        ...

    def forward(self, x):
        ...


class GPT(nn.Module):
    def __init__(self, vocab, block_size, d, n_layer, n_head):
        super().__init__()
        ...

    def forward(self, idx, targets=None):
        ...


torch.manual_seed(0)
m = GPT(vocab=20, block_size=16, d=32, n_layer=2, n_head=4)
logits, loss = m(torch.randint(0, 20, (2, 10)), torch.randint(0, 20, (2, 10)))
print(logits.shape, loss)   # (2, 10, 20), ≈ log 20 = 3.0

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Model hazır (`TinyGPT`, d aşamasındakiyle aynı yapı). Sen yalnız artımlı yolu yazıyorsun:

- `forward_step(model, idx_new, cache, pos)` — `idx_new: (B, t)` yeni token'lar, `pos` bunların ilk mutlak konumu, `cache` her blok için `(K, V)` ya da `None` (`(B, h, S, dk)`). Döndür: `(logits (B, t, V), yeni_cache)`. Her blokta yeni k, v'yi cache'e `torch.cat(…, dim=2)` ile ekle. Maske: $i$. yeni sorgu (mutlak konum `pos + i`) `0 … pos + i` key'lerini görür → `attn_mask = arange(S)[None] <= (pos + arange(t))[:, None]`. Tek döngü: bloklar.
- `generate_kv(model, idx, n)` — greedy; önce prompt'un tamamıyla **prefill**, sonra her adımda yalnız son token. Tek döngü: adımlar.

Kontrol, her konumdaki logit'lerin tam ileri geçişle aynı olduğunu ve üretilen dizinin cache'siz greedy ile birebir aynı olduğunu doğrular.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F
from torch import nn


class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(approximate="tanh"), nn.Linear(4 * d, d))

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(d, -1)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(o.transpose(1, 2).reshape(B, T, d))
        return x + self.mlp(self.ln2(x))


class TinyGPT(nn.Module):
    def __init__(self, V=64, T=256, d=64, L=4, h=4):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, d), nn.Embedding(T, d)
        self.blocks = nn.ModuleList(Block(d, h) for _ in range(L))
        self.ln_f = nn.LayerNorm(d)

    def forward(self, idx):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1]))
        for b in self.blocks:
            x = b(x)
        return self.ln_f(x) @ self.wte.weight.T


# ---- senin kodun ----

def forward_step(model, idx_new, cache, pos):
    """→ (logits (B, t, V), yeni cache). Tek döngü: bloklar."""
    ...


@torch.no_grad()
def generate_kv(model, idx, n):
    """Greedy; prefill + adım başına tek token. Tek döngü: adımlar."""
    ...


torch.manual_seed(0)
m = TinyGPT().eval()
print(generate_kv(m, torch.tensor([[1, 2, 3]]), 10))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Radford ve ark. (2019): GPT-2

[Radford, Wu, Child, Luan, Amodei, Sutskever — Language Models are Unsupervised Multitask Learners (OpenAI 2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

d aşamasındaki nanoGPT tarzı modeli GPT-2'nin mimari değişiklikleriyle hizala (Bölüm 2.3: Pre-LN, son LN, artık katmanların 1/√N ile başlatılması, 50 257'lik bayt düzeyli BPE) ve Hugging Face'teki GPT-2 small ağırlıklarını kendi modeline yükleyip aynı logit'leri üret. Sıfır örnekli görevlerden birini (ör. LAMBADA benzeri son kelime tahmini ya da 'TL;DR:' istemiyle özetleme, Bölüm 3) küçük bir örneklemde dene.


**Adımlar**

1. Ağırlık adlarını eşlerken Conv1D'nin transpoze ağırlık düzenine dikkat.

**Kabul ölçütleri**

- Yüklenen ağırlıklarla logit'ler HF ile 10⁻⁴ içinde.
- Sıfır örnekli görevden nitel örnekler.

## Makale ödevi: Kaplan ve ark. (2020): dil modelleri için ölçekleme yasaları

[Kaplan ve ark. — Scaling Laws for Neural Language Models (2020)](https://arxiv.org/abs/2001.08361)

Küçük ölçekte ölçekleme yasası çıkar: TinyStories ya da Shakespeare üzerinde 6–8 farklı boyutta (10⁵–10⁷ parametre, embedding hariç) model eğit; kaybı parametre sayısına ve hesaba (C ≈ 6ND) göre log-log çiz, L(N) = (N_c/N)^α_N uydur (makale Şekil 1). Erken durdurulmuş eğrilerden hesap-optimal sınırı çıkar ve Chinchilla (M8.1 ödevi) ile karşılaştır.


**Adımlar**

1. Her boyutta öğrenme oranını küçük bir taramayla ayarla.

**Kabul ölçütleri**

- Kayıp–N ilişkisi log-log'da yaklaşık doğrusal; α_N raporlanmış.
- Hesap-optimal sınır grafiği.

## Kaynaklar

- [Karpathy — nanoGPT ve Zero to Hero: Let's build GPT / Let's reproduce GPT-2](https://github.com/karpathy/nanoGPT) — MIT
- [Radford ve ark. — Language Models are Unsupervised Multitask Learners (GPT-2)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) — OpenAI
- [Stanford CS336 — Assignment 1 (Transformer LM from scratch)](https://github.com/stanford-cs336/assignment1-basics) — MIT
- [Harvard NLP — The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — MIT
- [Holtzman ve ark. — The Curious Case of Neural Text Degeneration (sampling)](https://arxiv.org/abs/1904.09751) — arXiv